In [2]:
import sys

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text, bindparam

from nhs_waiting_lists.utils.proj_paths import find_project_root

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [3]:

DB_PATH = project_root / "db/nhs_outp_activity_data.db"
DATA_DIR = "./data"

conn = create_engine(f"sqlite:///{DB_PATH}")

In [4]:
PROVIDER_CODES = ['R0B', 'RAJ', 'RTH', 'RTE', "RTF", "RWF", "RTE", "REF", "RWH", "R0B", "RVJ", "RHW",
                  "RDU", "RH8", "RWY", "RXC", "RL4", "RDE", "RXK", "RXR", "RJ2", "RN5", "RHU",
                  "RGN", "RWP", "RWD", "RAJ"]
TREATMENT_CODES = ('C_101', 'C_110', 'C_320', 'C_330', 'C_400', 'C_502', 'C_301')


In [9]:

query = text("""
             SELECT reporting_period,
                    geography_level,
                    organisation_code,
                    measure_type,
                    measure,
                    measure_value
             FROM outpatients_activity
    WHERE geography_level = 'Provider'
      AND measure_type = 'Attendance Type'
      AND (
          :provider_codes IS NULL
          OR organisation_code IN :provider_codes
      )
             ORDER BY geography_level ASC, organisation_code ASC, reporting_period ASC; \
             """).bindparams(
    bindparam('provider_codes', expanding=True),
)

df = pd.read_sql(query, conn, params={  # type: ignore[arg-type]
    'provider_codes': PROVIDER_CODES or None, }
                 )
df

OperationalError: (sqlite3.OperationalError) row value misused
[SQL: 
             SELECT reporting_period,
                    geography_level,
                    organisation_code,
                    measure_type,
                    measure,
                    measure_value
             FROM outpatients_activity
    WHERE geography_level = 'Provider'
      AND measure_type = 'Attendance Type'
      AND (
          (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?) IS NULL
          OR organisation_code IN (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
      )
             ORDER BY geography_level ASC, organisation_code ASC, reporting_period ASC;              ]
[parameters: ('R0B', 'RAJ', 'RTH', 'RTE', 'RTF', 'RWF', 'RTE', 'REF', 'RWH', 'R0B', 'RVJ', 'RHW', 'RDU', 'RH8', 'RWY', 'RXC', 'RL4', 'RDE', 'RXK', 'RXR', 'RJ2', 'RN5', 'RHU', 'RGN', 'RWP', 'RWD', 'RAJ', 'R0B', 'RAJ', 'RTH', 'RTE', 'RTF', 'RWF', 'RTE', 'REF', 

In [22]:
df_provider = df.query("organisation_code == 'RAJ'")

In [23]:
df_provider.drop_duplicates(subset=['measure'], keep='last').sort_values("measure")

,reporting_period,geography_level,organisation_code,measure_type,measure,measure_value
16,2425,03.Provider,RAJ,Attendance Type,01.Attended first appointment,377545.0
17,2425,03.Provider,RAJ,Attendance Type,02.Attended first tele consultation,61190.0
18,2425,03.Provider,RAJ,Attendance Type,03.Attended subsequent appointment,857480.0
19,2425,03.Provider,RAJ,Attendance Type,04.Attended subsequent tele consultation,246280.0
20,2425,03.Provider,RAJ,Attendance Type,06.Did not attend first appointment,34280.0
21,2425,03.Provider,RAJ,Attendance Type,07.Did not attend first tele consultation,690.0
22,2425,03.Provider,RAJ,Attendance Type,08.Did not attend subsequent appointment,71230.0
23,2425,03.Provider,RAJ,Attendance Type,09.Did not attend subsequent tele consultation,3010.0
24,2425,03.Provider,RAJ,Attendance Type,11.Patient cancelled first appointment,50325.0
25,2425,03.Provider,RAJ,Attendance Type,12.Patient cancelled first tele consultation,350.0
